# Error Analysis

Categorize wrong predictions with a simple taxonomy:
- **Arithmetic error** — set up correctly, miscalculated
- **Multi-step reasoning failure** — missed/confused reasoning steps
- **Quantity misinterpretation** — grabbed the wrong number from the question
- **Other** — no output, hallucination, etc.

Point `PRED_FILE` to whichever predictions JSON you want to analyze.

In [ ]:
import json
import re
import sys
from typing import Set

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, "..")
from src.evaluate import extract_predicted_answer, extract_ground_truth, is_correct

In [ ]:
PRED_FILE = "../results/prompted_preds.json"

with open(PRED_FILE) as f:
    preds = json.load(f)
print(f"Loaded {len(preds)} predictions")

## Overall accuracy

In [ ]:
rows = []
for rec in preds:
    pred_text = rec.get("raw_output") or rec.get("prediction", "")
    gold_text = rec.get("gold_answer", "")
    rows.append({
        "index": rec.get("index", 0),
        "question": rec.get("question", ""),
        "gold_answer": gold_text,
        "gold_value": extract_ground_truth(gold_text),
        "predicted_value": extract_predicted_answer(pred_text),
        "raw_output": pred_text,
        "correct": is_correct(pred_text, gold_text),
    })

df = pd.DataFrame(rows)
n_right = df["correct"].sum()
print(f"Accuracy: {n_right}/{len(df)} = {n_right/len(df):.4f}")

In [ ]:
wrong = df[~df["correct"]].copy()
print(f"{len(wrong)} wrong predictions")
wrong.head()

## Heuristic error classification

Automated heuristic to suggest a category. For the report we manually verified a sample.

In [ ]:
def _classify_error(row) -> str:
    pred, gold, q = row["predicted_value"], row["gold_value"], row["question"]
    if pred is None or gold is None:
        return "other"

    # did the model just parrot a number from the question?
    q_nums: Set[float] = {float(n) for n in re.findall(r"\d+", q)}
    if pred in q_nums and pred != gold:
        return "quantity_misinterpretation"

    # small relative error -> likely arithmetic slip
    if gold != 0 and abs(pred - gold) / abs(gold) < 0.25:
        return "arithmetic_error"

    return "multistep_failure"


wrong["error_type"] = wrong.apply(_classify_error, axis=1)
print(wrong["error_type"].value_counts())

In [ ]:
counts = wrong["error_type"].value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
ax.pie(counts.values, labels=counts.index, autopct="%1.1f%%", startangle=90)
ax.set_title("Error type breakdown")
plt.show()

## Inspect specific errors

In [ ]:
def _show_errors(df_wrong, etype: str, n: int = 3):
    sub = df_wrong[df_wrong["error_type"] == etype]
    print(f"\n--- {etype} ({len(sub)} total) ---")
    for _, row in sub.head(n).iterrows():
        print(f"\n[idx {row['index']}]")
        print(f"  Q: {row['question'][:150]}...")
        print(f"  Gold: {row['gold_value']}, Pred: {row['predicted_value']}")
        print(f"  Output: {str(row['raw_output'])[:150]}")

for et in wrong["error_type"].unique():
    _show_errors(wrong, et, n=2)